In [2]:
from pathlib import Path
import json
import re

import pandas as pd


def natural_sort_key(value):
    name = value.name if isinstance(value, Path) else str(value)
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", name)]


PROJECT_ROOT = Path.cwd().parent
DATASETS = ("L23", "L24", "L28")

for dataset_name in DATASETS:
    data_dir = PROJECT_ROOT / f"{dataset_name}_data"
    map_dir = data_dir / "map-keyframe"
    shot_dir = data_dir / "shot"
    output_keyframe_metadata = data_dir / "metadata.csv"
    output_shot_metadata = data_dir / "caption_metadata.csv"
    map_files = sorted(map_dir.glob(f"{dataset_name}_V*.csv"), key=natural_sort_key)

    if not map_files:
        raise FileNotFoundError(f"No map-keyframe files found in {map_dir}")

    keyframe_metadata_parts = []
    shot_metadata_parts = []

    for map_path in map_files:
        video_name = map_path.stem
        shot_path = shot_dir / f"{video_name}.json"
        if not shot_path.is_file():
            raise FileNotFoundError(f"Missing shot file for {video_name}: {shot_path}")

        mapping = pd.read_csv(map_path, usecols=["n", "pts_time", "frame_idx"])
        if mapping["n"].duplicated().any():
            raise ValueError(f"Duplicate n values in {map_path}")

        with shot_path.open("r", encoding="utf-8") as file:
            shots = json.load(file)
        if not isinstance(shots, list):
            raise ValueError(f"Expected a list of shots in {shot_path}")

        n_to_shot_id = {}
        shot_rows = []
        for shot in sorted(shots, key=lambda item: int(item["shot_id"])):
            shot_id = int(shot["shot_id"])
            keyframes = shot.get("keyframes", [])
            if not keyframes:
                raise ValueError(f"Shot {shot_id} has no keyframes in {shot_path}")

            first_keyframe = keyframes[0]
            last_keyframe = keyframes[-1]
            for keyframe in keyframes:
                keyframe_n = int(keyframe["n"])
                if keyframe_n in n_to_shot_id:
                    raise ValueError(f"Duplicate n={keyframe_n} in {shot_path}")
                n_to_shot_id[keyframe_n] = shot_id

            shot_rows.append(
                {
                    "video_name": video_name,
                    "shot_id": shot_id,
                    "start_frame_idx": int(first_keyframe["frame_idx"]),
                    "end_frame_idx": int(last_keyframe["frame_idx"]),
                    "start_time": float(shot["start_time"]),
                    "end_time": float(shot["end_time"]),
                }
            )

        map_n = set(mapping["n"].astype(int))
        shot_n = set(n_to_shot_id)
        if map_n != shot_n:
            missing_in_shot = sorted(map_n - shot_n)
            missing_in_map = sorted(shot_n - map_n)
            raise ValueError(
                f"n mismatch for {video_name}: "
                f"missing_in_shot={missing_in_shot[:5]}, "
                f"missing_in_map={missing_in_map[:5]}"
            )

        keyframe_metadata = mapping.assign(
            video_name=video_name,
            shot_id=mapping["n"].astype(int).map(n_to_shot_id),
        )[["video_name", "n", "frame_idx", "pts_time", "shot_id"]]
        keyframe_metadata_parts.append(keyframe_metadata)
        shot_metadata_parts.append(pd.DataFrame(shot_rows))

    metadata = pd.concat(keyframe_metadata_parts, ignore_index=True)
    caption_metadata = pd.concat(shot_metadata_parts, ignore_index=True)
    metadata.to_csv(output_keyframe_metadata, index=False)
    caption_metadata.to_csv(output_shot_metadata, index=False)

    print(f"Saved {output_keyframe_metadata}: shape={metadata.shape}")
    print(f"Saved {output_shot_metadata}: shape={caption_metadata.shape}")
    print(f"Keyframe columns: {list(metadata.columns)}")
    print(f"Shot columns: {list(caption_metadata.columns)}")

Saved d:\Truong_pipeline\My_Code\L23_data\metadata.csv: shape=(2326, 5)
Saved d:\Truong_pipeline\My_Code\L23_data\caption_metadata.csv: shape=(319, 6)
Keyframe columns: ['video_name', 'n', 'frame_idx', 'pts_time', 'shot_id']
Shot columns: ['video_name', 'shot_id', 'start_frame_idx', 'end_frame_idx', 'start_time', 'end_time']
Saved d:\Truong_pipeline\My_Code\L24_data\metadata.csv: shape=(6781, 5)
Saved d:\Truong_pipeline\My_Code\L24_data\caption_metadata.csv: shape=(753, 6)
Keyframe columns: ['video_name', 'n', 'frame_idx', 'pts_time', 'shot_id']
Shot columns: ['video_name', 'shot_id', 'start_frame_idx', 'end_frame_idx', 'start_time', 'end_time']
Saved d:\Truong_pipeline\My_Code\L28_data\metadata.csv: shape=(10683, 5)
Saved d:\Truong_pipeline\My_Code\L28_data\caption_metadata.csv: shape=(1002, 6)
Keyframe columns: ['video_name', 'n', 'frame_idx', 'pts_time', 'shot_id']
Shot columns: ['video_name', 'shot_id', 'start_frame_idx', 'end_frame_idx', 'start_time', 'end_time']
